In [ ]:
%load_ext autoreload
%autoreload 2

### Import dependencies

In [ ]:
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn import model_selection
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

In [ ]:
import data_prep
import ngram
import linguistic_feature

### Import Data

In [ ]:
df = data_prep.import_chi_review()
display(df[:5])

### Train-Test Split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=0)

### Preprocess & Feature Extraction

In [ ]:
df_train = data_prep.preprocess_ngram(df_train)
df_test = data_prep.preprocess_ngram(df_test)

In [ ]:
unigrams, bigrams, trigrams, unigrams_dict, bigrams_dict, trigrams_dict = ngram.find_universal_ngrams(df_train['PP_Review'], 16, 16, 10)

In [ ]:
df_train['Ngram'] = df_train['Word_List'].apply(lambda x: ngram.find_ngrams(x, unigrams, bigrams, trigrams))
df_test['Ngram'] = df_test['Word_List'].apply(lambda x: ngram.find_ngrams(x, unigrams, bigrams, trigrams))
display(df_train.head(3))
display(df_test.head(3))

In [ ]:
df_train = linguistic_feature.ling_feature(df_train)
df_test = linguistic_feature.ling_feature(df_test)

In [ ]:
bigram_words = [' '.join(bigram) for bigram in bigrams]
trigram_words = [' '.join(trigram) for trigram in trigrams]
new_words = unigrams + bigram_words + trigram_words
word2ind = dict(zip(new_words, range(len(new_words))))

In [ ]:
X_train, y_train = data_prep.df2matrix(df_train, word2ind)
X_test, y_test = data_prep.df2matrix(df_test, word2ind)

In [ ]:
X_train_pos = X_train[X_train['Rating'] == 5]
y_train_pos = y_train[X_train['Rating'] == 5]
X_train_neg = X_train[X_train['Rating'] == 1]
y_train_neg = y_train[X_train['Rating'] == 1]

X_test_pos = X_test[X_test['Rating'] == 5]
y_test_pos = y_test[X_test['Rating'] == 5]
X_test_neg = X_test[X_test['Rating'] == 1]
y_test_neg = y_test[X_test['Rating'] == 1]

In [ ]:
X_train_std = StandardScaler().fit_transform(X_train)
X_test_std = StandardScaler().fit_transform(X_test)

In [ ]:
X_train_pos_std = X_train_std[X_train['Rating'] == 5]
X_train_neg_std = X_train_std[X_train['Rating'] == 1]

X_test_pos_std = X_test_std[X_test['Rating'] == 5]
X_test_neg_std = X_test_std[X_test['Rating'] == 1]

### Train Model

#### 1) Random Forest

In [ ]:
rf_model = RandomForestClassifier(random_state=0)
rf_model.fit(X_train, y_train)

In [ ]:
rf_model = RandomForestClassifier(criterion='gini', random_state=0)
parameters = {'n_estimators': [100, 500],
              'max_features': [5, 10, 20],
              'max_depth': [20, 50],
              'max_samples': [100, 500, None]}
grid_search = GridSearchCV(rf_model, param_grid=parameters, cv=5, scoring='accuracy', verbose=2)
grid_search.fit(X_train, y_train)
print('Best Parameters: ', grid_search.best_params_)
print('Score', grid_search.best_score_)

In [ ]:
rf_model = RandomForestClassifier(random_state=0, 
                                  criterion='gini', 
                                  max_depth=20, 
                                  max_features=5, 
                                  max_samples=None, 
                                  n_estimators=500)
rf_model.fit(X_train, y_train)
print('Random Forest Performance')
print('- Train Overall Accuracy: ', "{:.1%}".format(rf_model.score(X_train, y_train)))
print('- Test Overall Accuracy: ', "{:.1%}".format(rf_model.score(X_test, y_test)))
print('- Test Positive Accuracy: ', "{:.1%}".format(rf_model.score(X_test_pos, y_test_pos)))
print('- Test Negative Accuracy: ', "{:.1%}".format(rf_model.score(X_test_neg, y_test_neg)))

#### 2) PCA + SVM

In [ ]:
pca = PCA()
svm_model = SVC()
pipeline = Pipeline(steps=[('pca', pca), ('svm', svm_model)])
parameters = {'svm__C': [3, 4], 'pca__n_components': list(range(1, 1001, 50))}
grid_search = GridSearchCV(pipeline, param_grid=parameters, cv=5, scoring='accuracy')
grid_search.fit(X_train_std, y_train)
print('Best Parameters: ', grid_search.best_params_)
print('Score', grid_search.best_score_)

In [ ]:
pca = PCA(n_components=601)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
X_test_pos_pca = pca.transform(X_test_pos_std)
X_test_neg_pca = pca.transform(X_test_neg_std)

svm_model = SVC(C=3)
svm_model.fit(X_train_pca, y_train)
print('SVM Performance')
print('- Train Overall Accuracy: ', "{:.1%}".format(svm_model.score(X_train_pca, y_train)))
print('- Test Overall Accuracy: ', "{:.1%}".format(svm_model.score(X_test_pca, y_test)))
print('- Test Positive Accuracy: ', "{:.1%}".format(svm_model.score(X_test_pos_pca, y_test_pos)))
print('- Test Negative Accuracy: ', "{:.1%}".format(svm_model.score(X_test_neg_pca, y_test_neg)))